## Downloading Dataset from Kaggle

In [ ]:
# import kagglehub

# path = kagglehub.dataset_download("harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows")

# print("Path to dataset files:", path)

## Imports

In [ ]:
import pandas as pd
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer

# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('averaged_perceptron_tagger')

## 1. Data Pre-processing

In [ ]:
# Loading Dataset
df = pd.read_csv(r"./dataset/imdb_top_1000.csv")
df.head(2)

In [ ]:
# Filtering Dataset

# Dropping unwanted columns
unwanted_columns = ["Poster_Link"]
df.drop(columns=unwanted_columns, inplace=True)

# Handling Missing Value
df["Meta_score"].fillna(df["Meta_score"].median(), inplace=True)
df["Gross"].fillna(0, inplace=True)
df["Certificate"].fillna("Unknown", inplace=True)

In [ ]:
# Formatting Data
text_columns = ["Series_Title", "Certificate", "Genre", "Overview", "Director", "Star1", "Star2", "Star3", "Star4"]
for col in text_columns:
    df[col] = df[col].str.lower()

df['Runtime'] = df['Runtime'].str.replace(' min', '').astype(int)

In [ ]:
# Tokenization
df["Temp"] = df[text_columns].astype(str).agg(" ".join, axis=1)
df["Tokens"] = df["Temp"].apply(word_tokenize)
df.drop(columns=["Temp"])

In [ ]:
# Filtering Tokens
punctuation = set(string.punctuation)
stop_words = set(stopwords.words('english'))

filter_set = stop_words.union(punctuation)

def filter_tokens(tkn):
    filtered_words = [word for word in tkn if word not in filter_set and word.isalpha()]
    return filtered_words

df["Filtered_Tokens"] = df["Tokens"].apply(filter_tokens)
df["Filtered_Tokens"].head()

In [ ]:
# Lemmatization
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN
    
def lemmatize_tokens(tkns):
    pos_tags = nltk.pos_tag(tkns)
    lemmatized_words = []
    
    for word, tag in pos_tags:
        wntag = get_wordnet_pos(tag)
        lemma = lemmatizer.lemmatize(word, wntag)
        lemmatized_words.append(lemma)
    
    return lemmatized_words

df["Lemmatized_Tokens"] = df["Filtered_Tokens"].apply(lemmatize_tokens)
df["Lemmatized_Tokens"].head()

## 2. Feature Extraction